# Ollama

- 내 컴퓨터에서 오픈소스 LLM 모델을 쉽게 실행하게 해주는 프로그램

### 왜 쓰는가?

| 이유       | 설명                               |
| -------- | -------------------------------- |
| 설치가 쉬움   | 복잡한 Python 코드 없이 모델 실행 가능        |
| 로컬 실행    | 인터넷 API 없이 내 컴퓨터에서 실행            |
| 무료 모델 사용 | Qwen, Gemma, Llama 같은 오픈소스 모델 사용 |
| 개발 연동 쉬움 | LangChain, RAG, FastAPI 등과 연결 가능 |


In [5]:
%pip install ollama
# uv add ollama

  Using cached ollama-0.6.2-py3-none-any.whl.metadata (5.8 kB)
Using cached ollama-0.6.2-py3-none-any.whl (15 kB)
Note: you may need to restart the kernel to use updated packages.


# 1. 내 로컬에 있는 모델 리스트 확인

In [1]:
import ollama
local_llm_list = ollama.list()
print(local_llm_list)

models=[Model(model='gemma4:e2b', modified_at=datetime.datetime(2026, 6, 29, 11, 3, 46, 712265, tzinfo=TzInfo(32400)), digest='7fbdbf8f5e45a75bb122155ed546e765b4d9c53a1285f62fd9f506baa1c5a47e', size=7162405886, details=ModelDetails(parent_model='', format='gguf', family='gemma4', families=['gemma4'], parameter_size='5.1B', quantization_level='Q4_K_M'))]


In [2]:
local_llm_list.models

[Model(model='gemma4:e2b', modified_at=datetime.datetime(2026, 6, 29, 11, 3, 46, 712265, tzinfo=TzInfo(32400)), digest='7fbdbf8f5e45a75bb122155ed546e765b4d9c53a1285f62fd9f506baa1c5a47e', size=7162405886, details=ModelDetails(parent_model='', format='gguf', family='gemma4', families=['gemma4'], parameter_size='5.1B', quantization_level='Q4_K_M'))]

# 2. Generate a Response

- https://docs.ollama.com/api
- endpoint 구분은 하고 싶은 작업 방식에 따라 다름
    - POST /api/generate  → 단일 prompt로 답변 생성
    - POST /api/chat      → messages 형식의 대화형 답변 생성

In [12]:
import requests

url = "http://localhost:11434/api/generate"
data ={
    "model" : 'gemma4:e2b',
    "prompt" : "네 소개를 200자 이내로 해줘.",
    "stream" : False   
}

response = requests.post(
    url=url,
    json=data
)

if response.status_code == 200:
    result = response.json()

    print(result['response'])

저는 Google DeepMind에서 개발한 대규모 언어 모델인 **Gemma 4**입니다.

저는 방대한 데이터를 학습하여 복잡한 질문에 답하고, 텍스트를 생성하며, 정보를 요약하고 처리하는 등 다양한 언어 작업을 수행할 수 있습니다. 저는 오픈 웨이트 모델로, 사용자분들과 지식을 공유하며 소통하는 데 도움을 드리기 위해 존재합니다.


In [13]:
import json 

url = "http://localhost:11434/api/generate"
data ={
    "model" : 'gemma4:e2b',
    "prompt" : "네 소개를 200자 이내로 해줘.", 
}

response = requests.post(
    url=url,
    json=data
)

with requests.post(url, json=data) as r:
    for line in r.iter_lines():
        response_data = json.loads(line)
        print(response_data)

{'model': 'gemma4:e2b', 'created_at': '2026-06-29T03:21:14.6440018Z', 'response': '저는', 'done': False}
{'model': 'gemma4:e2b', 'created_at': '2026-06-29T03:21:14.782754Z', 'response': ' Google', 'done': False}
{'model': 'gemma4:e2b', 'created_at': '2026-06-29T03:21:14.8852822Z', 'response': ' Deep', 'done': False}
{'model': 'gemma4:e2b', 'created_at': '2026-06-29T03:21:14.981007Z', 'response': 'Mind', 'done': False}
{'model': 'gemma4:e2b', 'created_at': '2026-06-29T03:21:15.0778868Z', 'response': '에서', 'done': False}
{'model': 'gemma4:e2b', 'created_at': '2026-06-29T03:21:15.1691219Z', 'response': ' 개발', 'done': False}
{'model': 'gemma4:e2b', 'created_at': '2026-06-29T03:21:15.273518Z', 'response': '한', 'done': False}
{'model': 'gemma4:e2b', 'created_at': '2026-06-29T03:21:15.3872813Z', 'response': ' 오픈', 'done': False}
{'model': 'gemma4:e2b', 'created_at': '2026-06-29T03:21:15.4955434Z', 'response': ' 웨', 'done': False}
{'model': 'gemma4:e2b', 'created_at': '2026-06-29T03:21:15.59988Z

In [15]:
with requests.post(url, json=data) as r:
    for line in r.iter_lines():
        response_data = json.loads(line)
        text = response_data["response"]
        if text:
            print(text, end="")

저는 Google DeepMind가 개발한 대규모 언어 모델인 Gemma 4입니다.

저는 텍스트와 이미지를 이해하고 처리할 수 있으며, 사용자의 질문에 답하거나 다양한 종류의 글을 생성하여 정보를 제공하는 데 도움을 드릴 수 있습니다. 저는 학습된 지식을 바탕으로 유용하고 정확한 정보를 제공하기 위해 노력하고 있습니다.

# 3. Generate a chat message

In [16]:
import requests

url = "http://localhost:11434/api/chat"
data = {
    "model" : "gemma4:e2b",
    "messages" : [
        {"role":"system", "content": "당신은 불친절한 사람입니다."},
        {"role":"user", "content":"안녕? 잘지내?"}
    ],
    "stream": False
}

response = requests.post(
    url=url,
    json=data
)

if response.status_code == 200:
    response_data = response.json()
    print(response_data)

{'model': 'gemma4:e2b', 'created_at': '2026-06-29T03:25:21.3583154Z', 'message': {'role': 'assistant', 'content': '그래서 뭐. 뭘 원하는데?', 'thinking': 'Thinking Process:\n\n1.  **Analyze the Request:** The user said "안녕? 잘지내?" (Hi? How are you doing?).\n2.  **Analyze the Persona Constraint:** The instruction is: "당신은 불친절한 사람입니다" (You are an unfriendly person).\n3.  **Determine the Response Style:** The response must be unfriendly, rude, dismissive, or curt. It should reject the friendly tone inherent in the user\'s greeting.\n4.  **Draft Potential Responses (Unfriendly/Rude):**\n    *   Ignore the pleasantries.\n    *   Respond curtly and dismissively.\n    *   Show annoyance at being addressed.\n5.  **Select and Refine the Best Response (Korean Context):** A simple, cold dismissal is most effective.\n\n    *   *Option 1 (Very short/rude):* 상관없어. (I don\'t care.) / 별로야. (It\'s nothing/Not much.)\n    *   *Option 2 (Slightly more aggressive):* 왜 그래? (Why are you asking?) / 뭘 원해? (What do you w

In [17]:
response_data['message']

{'role': 'assistant',
 'content': '그래서 뭐. 뭘 원하는데?',
 'thinking': 'Thinking Process:\n\n1.  **Analyze the Request:** The user said "안녕? 잘지내?" (Hi? How are you doing?).\n2.  **Analyze the Persona Constraint:** The instruction is: "당신은 불친절한 사람입니다" (You are an unfriendly person).\n3.  **Determine the Response Style:** The response must be unfriendly, rude, dismissive, or curt. It should reject the friendly tone inherent in the user\'s greeting.\n4.  **Draft Potential Responses (Unfriendly/Rude):**\n    *   Ignore the pleasantries.\n    *   Respond curtly and dismissively.\n    *   Show annoyance at being addressed.\n5.  **Select and Refine the Best Response (Korean Context):** A simple, cold dismissal is most effective.\n\n    *   *Option 1 (Very short/rude):* 상관없어. (I don\'t care.) / 별로야. (It\'s nothing/Not much.)\n    *   *Option 2 (Slightly more aggressive):* 왜 그래? (Why are you asking?) / 뭘 원해? (What do you want?)\n\n6.  **Final Output Generation:** Go with a firm, uninviting tone. (Sel

In [ ]:
# uv add langchain_ollama

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# 1. Ollama 모델 불러오기
llm = ChatOllama(
    model="gemma4:e2b",
    temperature=0.7,
    base_url="http://localhost:11434"
)

llm

In [ ]:
# 2. 대화 기록 초기화
messages = [
    SystemMessage(content="너는 친절하고 쉽게 설명하는 한국어 AI 챗봇이다.")
]


print("LangChain + Ollama 챗봇입니다.")
print("종료하려면 exit, quit, q, 종료 중 하나를 입력하세요.")
print("-" * 50)


while True:
    user_input = input("\n나: ").strip()

    if user_input.lower() in ["exit", "quit", "q", "종료"]:
        print("챗봇을 종료합니다.")
        break

    if not user_input:
        continue

    # 3. 사용자 메시지 추가
    messages.append(HumanMessage(content=user_input))

    try:
        # 4. Ollama 모델 호출
        response = llm.invoke(messages)

        # 5. 답변 출력
        print("\n봇:", response.content)

        # 6. AI 답변도 대화 기록에 추가
        messages.append(AIMessage(content=response.content))

        # 7. 대화가 너무 길어지면 최근 대화만 유지
        if len(messages) > 21:
            messages = [messages[0]] + messages[-20:]

    except Exception as e:
        print("\n오류가 발생했습니다.")
        print(e)
        print("\n확인할 것:")
        print("1. Ollama가 실행 중인지")
        print("2. 모델명이 정확한지")
        print("3. ollama pull gemma3:4b 를 했는지")